# Exploratory audit: S&P 500 membership history

Inspect CSV completeness, identifiers, dates, and historical coverage.
See [audit findings](exploratory_data_audit.md) for provenance and dated results.

Select the `indexlens` kernel and run all cells from the project root or `exploratory_data_audit/`.
Each run downloads the latest source CSV into `data/raw/components_history.csv`.
Audit calculations leave the loaded values unchanged.

## 1. Download and load

Download the components history CSV from [lawcal/sp500-components-history](https://github.com/lawcal/sp500-components-history).
Load columns as strings to preserve identifiers; default missing-value recognition applies.

In [62]:
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

SOURCE_URL = (
    "https://raw.githubusercontent.com/lawcal/sp500-components-history/"
    "main/data/components_history.csv"
)
if (Path.cwd() / "pyproject.toml").is_file():
    DATA_PATH = Path("data/raw/components_history.csv")
else:
    DATA_PATH = Path("../data/raw/components_history.csv")

response = requests.get(SOURCE_URL, timeout=30)
response.raise_for_status()
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
DATA_PATH.write_bytes(response.content)

df = pd.read_csv(DATA_PATH, dtype="string")
df.shape

(1002, 7)

### Columns and preview

All seven expected columns must be present. Extra columns are reported separately.

In [63]:
expected_columns = [
    "symbol",
    "cik",
    "name",
    "sector",
    "date_added",
    "date_removed",
    "created_at",
]
missing_columns = [column for column in expected_columns if column not in df.columns]
extra_columns = [column for column in df.columns if column not in expected_columns]
display(pd.Series({"missing_columns": missing_columns, "extra_columns": extra_columns}))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")
if len(df) == 0:
    raise ValueError("The CSV contains no records.")
display(df.dtypes)
display(df.head(5))
display(df.tail(5))

missing_columns    []
extra_columns      []
dtype: object

symbol          string
cik             string
name            string
sector          string
date_added      string
date_removed    string
created_at      string
dtype: object

,symbol,cik,name,sector,date_added,date_removed,created_at
0,A,0001090872,Agilent Technologies,health_care,2000-06-05,<NA>,2007-03-05
1,AA,0000004281,Alcoa Inc,materials,1964-03-31,2016-11-01,2007-03-05
2,AAL,0000006201,American Airlines Group,industrials,2015-03-23,2024-09-23,2015-03-20
3,AAP,0001158449,Advance Auto Parts,consumer_discretionary,2015-07-09,2023-08-25,2015-07-09
4,AAPL,0000320193,Apple Inc.,information_technology,1982-11-30,<NA>,2007-03-05


,symbol,cik,name,sector,date_added,date_removed,created_at
997,ZBH,0001136869,Zimmer Biomet,health_care,2001-08-07,<NA>,2015-06-29
998,ZBRA,0000877212,Zebra Technologies,information_technology,2019-12-23,<NA>,2019-12-31
999,ZION,0000109380,Zions Bancorporation,financials,2001-06-22,2024-03-18,2007-03-05
1000,ZMH,0001136869,Zimmer Holdings,health_care,2007-03-05*,2015-06-29*,2007-03-05
1001,ZTS,0001555280,Zoetis,health_care,2013-06-21,<NA>,2013-08-23


## 2. Completeness and text

Count missing values separately from whitespace-only strings. A missing removal
date does not establish current membership.

In [64]:
null_report = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100,
    }
).sort_values("missing_count", ascending=False)
display(null_report)

required_fields = [column for column in expected_columns if column != "date_removed"]
blank_counts = df[expected_columns].apply(
    lambda column: column.str.strip().eq("").sum()
)
whitespace_counts = df[expected_columns].apply(
    lambda column: column.ne(column.str.strip()).sum()
)
display(
    pd.DataFrame(
        {
            "blank_count": blank_counts,
            "surrounding_whitespace_count": whitespace_counts,
        }
    )
)

,missing_count,missing_percent
date_removed,503,50.199601
symbol,0,0.000000
cik,0,0.000000
name,0,0.000000
sector,0,0.000000
date_added,0,0.000000
created_at,0,0.000000


,blank_count,surrounding_whitespace_count
symbol,0,0
cik,0,0
name,0,0
sector,0,0
date_added,0,0
date_removed,0,0
created_at,0,0


In [65]:
non_uppercase_symbols = df["symbol"].ne(df["symbol"].str.upper())
display(pd.Series({"non_uppercase_symbols": non_uppercase_symbols.sum()}))
display(df["symbol"].str.len().value_counts().sort_index())

longest_symbol_length = df["symbol"].str.len().max()
display(df.loc[df["symbol"].str.len() == longest_symbol_length])

non_uppercase_symbols    0
dtype: int64

symbol
1     19
2     99
3    591
4    286
5      6
6      1
Name: count, dtype: Int64

,symbol,cik,name,sector,date_added,date_removed,created_at
799,SGPPRB,0000310158,Schering-Plough,health_care,2009-12-31*,2010-01-22*,2009-12-31


Inspect symbol-length outliers without imposing an unsupported length limit.

## 3. Duplicates and identifiers

`duplicated()` counts exclude the first occurrence, e.g., `A, A, A, B` contains two repeated symbols.
`duplicated(keep=False)` selects every row in a repeated group for inspection.
Repeated symbols and CIKs are not automatically errors. A CIK is not assumed to
identify a unique security.


In [66]:
duplicate_counts = pd.Series(
    {
        "exact_duplicate_rows": df.duplicated().sum(),
        "repeated_symbols": df.duplicated(subset=["symbol"]).sum(),
        "repeated_ciks": df.duplicated(subset=["cik"]).sum(),
        "duplicate_symbol_date_pairs": df.duplicated(
            subset=["symbol", "date_added"]
        ).sum(),
        "duplicate_cik_symbol_date_groups": df.duplicated(
            subset=["cik", "symbol", "date_added"]
        ).sum(),
    }
)
display(duplicate_counts)
display(df[["symbol", "cik", "sector"]].nunique())

exact_duplicate_rows                  0
repeated_symbols                     34
repeated_ciks                       106
duplicate_symbol_date_pairs           0
duplicate_cik_symbol_date_groups      0
dtype: int64

symbol    968
cik       896
sector     11
dtype: int64

In [67]:
repeated_symbols = df.loc[df["symbol"].duplicated(keep=False)]
repeated_ciks = df.loc[df["cik"].duplicated(keep=False)]
display(repeated_symbols.sort_values(["symbol", "date_added"]).head(10))
display(repeated_ciks.sort_values(["cik", "symbol", "date_added"]).head(10))
display(df["sector"].value_counts())

,symbol,cik,name,sector,date_added,date_removed,created_at
13,ACE,0000896159,ACE Limited,financials,2007-03-05*,2008-08-21*,2007-03-05
14,ACE,0000896159,ACE Limited,financials,2010-07-15,2016-01-19,2010-07-21
32,AGN,0000850693,Allergan Inc,health_care,1992-02-07,2015-03-23,2007-03-05
33,AGN,0001578845,"Allergan, plc",health_care,1999-04-12,2020-05-12,2015-06-20
49,AMD,0000002488,Advanced Micro Devices,information_technology,2007-03-05*,2013-09-20,2007-03-05
50,AMD,0000002488,Advanced Micro Devices,information_technology,2017-03-20,<NA>,2017-03-18
158,CBE,0001141982,Cooper Industries Ltd.,industrials,2007-03-05*,2009-09-28,2007-03-05
159,CBE,0001141982,Cooper Industries,industrials,2011-11-30*,2012-12-03,2011-11-30
175,CEG,0001004440,Constellation Energy Group,utilities,2007-03-05*,2012-03-13,2007-03-05
176,CEG,0001868275,Constellation Energy,utilities,2022-02-02,<NA>,2022-02-07


,symbol,cik,name,sector,date_added,date_removed,created_at
49,AMD,0000002488,Advanced Micro Devices,information_technology,2007-03-05*,2013-09-20,2007-03-05
50,AMD,0000002488,Advanced Micro Devices,information_technology,2017-03-20,<NA>,2017-03-18
1,AA,0000004281,Alcoa Inc,materials,1964-03-31,2016-11-01,2007-03-05
79,ARNC,0000004281,Arconic Inc.,industrials,1964-03-31,2020-04-01,2016-11-01
464,HWM,0000004281,Howmet Aerospace,industrials,2016-10-21,<NA>,2020-04-02
101,BALL,0000009389,Ball Corporation,materials,1984-10-31,<NA>,2022-05-10
128,BLL,0000009389,Ball,materials,1984-10-31,2022-05-10*,2007-03-05
237,CTL,0000018926,CenturyLink Inc,communication_services,1999-03-25,2020-09-22*,2007-03-05
563,LUMN,0000018926,Lumen Technologies,communication_services,1999-03-25,2023-03-20,2020-09-22
251,CZN,0000020520,Citizens Communications,communication_services,2007-03-05*,2008-08-21*,2007-03-05


sector
information_technology    141
financials                141
industrials               135
consumer_discretionary    135
health_care               120
energy                     70
consumer_staples           63
materials                  54
communication_services     53
utilities                  46
real_estate                44
Name: count, dtype: Int64

Observed composite uniqueness describes this file, not a universal key.

## 4. Dates and uncertainty

The [source implementation](https://github.com/lawcal/sp500-components-history/blob/main/bin/update.py)
marks approximate dates with `*`. Missing effective dates can use the Wikipedia
revision date in New York time. Preserve flags and separate missing inputs from
malformed values. Successful parsing does not remove uncertainty.

In [68]:
added_circa = df["date_added"].str.endswith("*", na=False)
removed_circa = df["date_removed"].str.endswith("*", na=False)
added_clean = df["date_added"].str.removesuffix("*")
removed_clean = df["date_removed"].str.removesuffix("*")

parsed_added = pd.to_datetime(df["date_added"], format="%Y-%m-%d", errors="coerce")
parsed_removed = pd.to_datetime(df["date_removed"], format="%Y-%m-%d", errors="coerce")
parsed_created = pd.to_datetime(df["created_at"], format="%Y-%m-%d", errors="coerce")
parsed_added_clean = pd.to_datetime(added_clean, format="%Y-%m-%d", errors="coerce")
parsed_removed_clean = pd.to_datetime(removed_clean, format="%Y-%m-%d", errors="coerce")

malformed_added = df["date_added"].notna() & parsed_added_clean.isna()
malformed_removed = df["date_removed"].notna() & parsed_removed_clean.isna()
malformed_created = df["created_at"].notna() & parsed_created.isna()

date_report = pd.DataFrame(
    {
        "missing": [
            df["date_added"].isna().sum(),
            df["date_removed"].isna().sum(),
            df["created_at"].isna().sum(),
        ],
        "approximate": [added_circa.sum(), removed_circa.sum(), 0],
        "unparsed_before_cleanup": [
            (df["date_added"].notna() & parsed_added.isna()).sum(),
            (df["date_removed"].notna() & parsed_removed.isna()).sum(),
            malformed_created.sum(),
        ],
        "unparsed_after_cleanup": [
            malformed_added.sum(),
            malformed_removed.sum(),
            malformed_created.sum(),
        ],
    },
    index=["date_added", "date_removed", "created_at"],
)
display(date_report)
display(df.loc[malformed_added | malformed_removed | malformed_created])

,missing,approximate,unparsed_before_cleanup,unparsed_after_cleanup
date_added,0,257,257,0
date_removed,503,128,128,0
created_at,0,0,0,0


,symbol,cik,name,sector,date_added,date_removed,created_at


In [69]:
# Inspect raw date text alongside parsed values and uncertainty flags.
date_examples = pd.DataFrame(
    {
        "symbol": df["symbol"],
        "date_added_raw": df["date_added"],
        "date_added_parsed": parsed_added_clean,
        "date_added_approximate": added_circa,
        "date_removed_raw": df["date_removed"],
        "date_removed_parsed": parsed_removed_clean,
        "date_removed_approximate": removed_circa,
    }
)
date_examples.loc[added_circa | removed_circa].head(5)

,symbol,date_added_raw,date_added_parsed,date_added_approximate,date_removed_raw,date_removed_parsed,date_removed_approximate
6,ABC,2001-08-30,2001-08-30,False,2023-09-01*,2023-09-01,True
7,ABI,2007-03-05*,2007-03-05,True,2009-03-07*,2009-03-07,True
8,ABK,2007-03-05*,2007-03-05,True,2008-06-10,2008-06-10,False
12,ACAS,2008-01-22*,2008-01-22,True,2009-03-03,2009-03-03,False
13,ACE,2007-03-05*,2007-03-05,True,2008-08-21*,2008-08-21,True


### Interval order and date ranges

Compare parseable endpoints using nominal dates. Approximate dates do not prove
real-world event order. Observed extrema do not establish complete coverage.

In [70]:
comparable_intervals = parsed_added_clean.notna() & parsed_removed_clean.notna()
invalid_interval_mask = comparable_intervals & (
    parsed_removed_clean < parsed_added_clean
)
equal_endpoint_mask = comparable_intervals & (
    parsed_removed_clean == parsed_added_clean
)
invalid_intervals = df.loc[invalid_interval_mask]

display(
    pd.Series(
        {
            "comparable_intervals": comparable_intervals.sum(),
            "removed_before_added": invalid_interval_mask.sum(),
            "equal_endpoints": equal_endpoint_mask.sum(),
        }
    )
)
display(invalid_intervals)

date_ranges = pd.DataFrame(
    {
        "earliest": [
            parsed_added_clean.min(),
            parsed_removed_clean.min(),
            parsed_created.min(),
        ],
        "latest": [
            parsed_added_clean.max(),
            parsed_removed_clean.max(),
            parsed_created.max(),
        ],
    },
    index=["date_added", "date_removed", "created_at"],
)
display(date_ranges)

comparable_intervals    499
removed_before_added      0
equal_endpoints           0
dtype: int64

,symbol,cik,name,sector,date_added,date_removed,created_at


,earliest,latest
date_added,1957-03-04,2026-09-21
date_removed,2007-03-30,2026-09-20
created_at,2007-03-05,2026-09-20


## 5. Historical coverage

The [source documentation](https://github.com/lawcal/sp500-components-history#data-source-caveats)
supports queries from 2007-03-05. Earlier additions do not establish earlier coverage.
`created_at` records when a symbol was first captured from the Wikipedia revisions;
it is not proof that every field was historically available then.

`first_created` is the earliest `created_at`. The suffix `_mask` identifies a
True/False condition for each row; `_rows` identifies the selected records.


In [71]:
first_created = parsed_created.min()
first_created_mask = parsed_created == first_created
first_created_rows = df.loc[first_created_mask]

display(
    pd.Series(
        {
            "first_created": first_created.strftime("%Y-%m-%d"),
            "Rows with created_at = first_created": first_created_mask.sum(),
            "Rows with date_added < first_created": (
                parsed_added_clean < first_created
            ).sum(),
            "Created at & added before first_created": (
                first_created_mask & (parsed_added_clean < first_created)
            ).sum(),
            "Created at & added on first_created": (
                first_created_mask & (parsed_added_clean == first_created)
            ).sum(),
            "Created at & added after first_created": (
                first_created_mask & (parsed_added_clean > first_created)
            ).sum(),
        }
    )
)

first_created                              2007-03-05
Rows with created_at = first_created              500
Rows with date_added < first_created              316
Created at & added before first_created           280
Created at & added on first_created               210
Created at & added after first_created             10
dtype: object

### Membership at `first_created`

The upstream dated-query helper uses inclusive addition, exclusive removal, and
`created_at <= query date`. Only source-missing removals count as open-ended here.
Approximate dates retain uncertainty even when the filter matches source conventions.

In [72]:
active_at_first_created_mask = (
    parsed_added_clean.notna()
    & (parsed_added_clean <= first_created)
    & (df["date_removed"].isna() | (parsed_removed_clean > first_created))
    & parsed_created.notna()
    & (parsed_created <= first_created)
)
active_at_first_created_rows = df.loc[active_at_first_created_mask]
excluded_first_created_mask = first_created_mask & ~active_at_first_created_mask

display(
    pd.Series(
        {
            "Rows with created_at = first_created": first_created_mask.sum(),
            "Active at first_created": active_at_first_created_mask.sum(),
            "Created at first_created: excluded": excluded_first_created_mask.sum(),
            "Active at first_created: approximate dates": (
                active_at_first_created_mask & (added_circa | removed_circa)
            ).sum(),
        }
    )
)

# Reason flags can overlap, so their counts are not additive.
first_created_exclusions = pd.DataFrame(
    {
        "date_added > first_created": parsed_added_clean > first_created,
        "date_removed <= first_created": parsed_removed_clean <= first_created,
        "date_added missing or malformed": parsed_added_clean.isna(),
        "date_removed malformed": malformed_removed,
    }
)
display(first_created_exclusions.loc[excluded_first_created_mask].sum())
display(df.loc[excluded_first_created_mask].head(10))

Rows with created_at = first_created          500
Active at first_created                       490
Created at first_created: excluded             10
Active at first_created: approximate dates    228
dtype: int64

date_added > first_created         10
date_removed <= first_created       0
date_added missing or malformed     0
date_removed malformed              0
dtype: int64

,symbol,cik,name,sector,date_added,date_removed,created_at
157,CB,0000896159,Chubb Limited,financials,2010-07-15,<NA>,2007-03-05
253,D,0000715957,Dominion Energy,utilities,2016-11-30,<NA>,2007-03-05
352,FCX,0000831259,Freeport-McMoRan,materials,2011-07-01,<NA>,2007-03-05
393,GAS,0001004155,AGL Resources Inc.,utilities,2011-12-13,2016-07-01,2007-03-05
415,GOOG,0001652044,Alphabet Inc. (Class C),communication_services,2014-04-03,<NA>,2007-03-05
463,HUM,0000049071,Humana,health_care,2012-12-10,<NA>,2007-03-05
497,JCI,0000833444,Johnson Controls,industrials,2010-08-27,<NA>,2007-03-05
675,NWSA,0001564708,News Corp (Class A),communication_services,2013-08-01,<NA>,2007-03-05
827,SRE,0001032208,Sempra,utilities,2017-03-17,<NA>,2007-03-05
883,TROW,0001113169,T. Rowe Price,financials,2019-07-29,<NA>,2007-03-05


The exclusion table identifies which conditions remove rows with `created_at = first_created`.
Matching the upstream filter does not resolve inconsistent source dates or establish
an independently verified historical membership count.

## 6. Summary

Calculated results describe this input file. Reassess written findings when the
dataset changes.

In [73]:
audit_summary = pd.Series(
    {
        "rows": len(df),
        "columns": len(df.columns),
        "missing_required_values": df[required_fields].isna().sum().sum(),
        "blank_required_values": blank_counts[required_fields].sum(),
        "surrounding_whitespace_values": whitespace_counts.sum(),
        "exact_duplicate_rows": duplicate_counts["exact_duplicate_rows"],
        "duplicate_symbol_date_pairs": duplicate_counts["duplicate_symbol_date_pairs"],
        "missing_removal_dates": df["date_removed"].isna().sum(),
        "approximate_addition_dates": added_circa.sum(),
        "approximate_removal_dates": removed_circa.sum(),
        "malformed_non_null_dates_after_cleanup": (
            malformed_added.sum() + malformed_removed.sum() + malformed_created.sum()
        ),
        "invalid_intervals": invalid_interval_mask.sum(),
        "first_created_rows": first_created_mask.sum(),
        "active_at_first_created_rows": active_at_first_created_mask.sum(),
    }
)
audit_summary

rows                                      1002
columns                                      7
missing_required_values                      0
blank_required_values                        0
surrounding_whitespace_values                0
exact_duplicate_rows                         0
duplicate_symbol_date_pairs                  0
missing_removal_dates                      503
approximate_addition_dates                 257
approximate_removal_dates                  128
malformed_non_null_dates_after_cleanup       0
invalid_intervals                            0
first_created_rows                         500
active_at_first_created_rows               490
dtype: int64

### Candidate validation rules

- Require expected columns and non-missing, non-blank required values.
- Separate missing removals from malformed dates; preserve raw text and uncertainty.
- Flag exact duplicates and reversed nominal intervals.
- Allow repeated symbols and CIKs; assess composite uniqueness against source semantics.

### Limitations

Wikipedia-derived history can contain omissions, corrections, and overlapping ticker
histories. Approximate dates, the initial-group discrepancy, equal endpoints, and
uniqueness after normalization require explicit interpretation.